# DSDBA — Deepfake Speech Detection & Biometric Authentication System
**Document:** DSDBA-SRS-2026-002 v2.1 | Phase 3 — Sprint B Training Notebook  
**Authors:** Ferel, Safa — ITS Informatics | KCVanguard ML Workshop  
**Runtime:** Google Colab Free Tier — T4 GPU (15 GB VRAM)  
**Status:** ACTIVE — training loop + ONNX export populated in Chain 06 (Sprint B)  

---

## Notebook Structure
| Cell | Purpose | SRS Ref |
|------|---------|--------|
| 1 | Environment setup — install pinned dependencies | NFR-Security |
| 2 | GPU verification — assert CUDA availability + print VRAM | Q3 |
| 3 | **Q3 VRAM Test** — EfficientNet-B4 peak VRAM at batch_size=16 | Q3 gate |
| 4 | HuggingFace login — authenticate + verify FoR dataset access | FR-CV-007 |
| 5 | Dataset download — FoR for-2sec split via datasets API | FR-AUD-004 |
| 6 | Training loop — two-phase EfficientNet-B4 training via `run_training()` | FR-CV-003–008 |
| 7 | ONNX export + equivalence verification | FR-DEP-010 |
| 8 | Upload best checkpoint to HF Hub | FR-CV-007 |
| 9 | Print EER and AUC-ROC results — verify targets | FR-CV-008 |

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 1 — Environment Setup
# SRS: NFR-Security (all deps pinned), FR-AUD-001–011, FR-CV-001–016, FR-DEP-010
# Updated 2026-03-27: numpy unpinned — Colab 2026 ships numpy 2.x and its
#   pre-built scikit-learn/scipy are compiled against 2.x; pinning 1.26.4
#   breaks _blas_supports_fpe and other numpy 2.x-only symbols at import time.
# ─────────────────────────────────────────────────────────────────────────────

# Step 0: confirm numpy version before any installs (informational)
import numpy as _np; print(f'Colab baseline numpy: {_np.__version__}'); del _np

# Core ML framework (torch 2.4.1 ↔ torchvision 0.19.1 ↔ torchaudio 2.4.1)
!pip install --quiet torch==2.4.1 torchvision==0.19.1 torchaudio==2.4.1

# Audio DSP — numpy intentionally unpinned; keep whatever Colab ships (≥2.0)
!pip install --quiet librosa==0.10.1 resampy==0.4.2 soundfile==0.12.1

# Computer Vision / XAI
!pip install --quiet grad-cam==1.5.0 Pillow==10.4.0 matplotlib==3.9.2 scikit-learn==1.5.2

# ONNX Runtime
!pip install --quiet onnx==1.17.0 onnxruntime==1.19.2

# NLP / API (aiohttp ≥3.9 for pre-built Python 3.12 wheels)
!pip install --quiet openai==1.3.0 httpx==0.25.0 aiohttp==3.10.10

# HuggingFace ecosystem
!pip install --quiet huggingface-hub==0.25.2 datasets==2.21.0

# Configuration & utilities
!pip install --quiet PyYAML==6.0.2 pydantic==2.9.2 python-dotenv==1.0.0

import numpy as _np; print(f'numpy after setup   : {_np.__version__}'); del _np
print('All dependencies installed successfully.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 2 — GPU Verification
# SRS: Q3 gate prerequisite — confirm T4 GPU available before VRAM test
# ─────────────────────────────────────────────────────────────────────────────

import torch

assert torch.cuda.is_available(), (
    'FATAL: CUDA not available. '
    'Go to Runtime → Change runtime type → GPU (T4) and restart.'
)

device_name = torch.cuda.get_device_name(0)
total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)

print(f'GPU device : {device_name}')
print(f'Total VRAM : {total_vram_gb:.2f} GB')
print(f'CUDA version: {torch.version.cuda}')
print(f'PyTorch version: {torch.__version__}')

assert total_vram_gb >= 14.0, (
    f'WARNING: VRAM {total_vram_gb:.2f} GB < 14 GB. '
    'Colab may have assigned a smaller GPU. Reduce batch_size if Q3 VRAM test fails.'
)

print('\n✅ GPU verification passed. Proceeding to Q3 VRAM test.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 3 — Q3 VRAM Test: EfficientNet-B4 peak memory at batch_size=16
# SRS: FR-CV-003 (training), Q3 gate — MUST pass before Sprint B begins
# Gate rule: If peak > 11 GB → gradient_checkpointing=true (config.yaml already set)
# MCP: context7-mcp (torchvision EfficientNet-B4)
# ─────────────────────────────────────────────────────────────────────────────

import torch
import torchvision

# ── Config values (from config.yaml) ─────────────────────────────────────────
BATCH_SIZE = 16           # config.yaml: training.batch_size
INPUT_H    = 224          # config.yaml: audio.output_tensor_shape[1]
INPUT_W    = 224          # config.yaml: audio.output_tensor_shape[2]
INPUT_C    = 3            # config.yaml: audio.output_tensor_shape[0]
VRAM_THRESHOLD_GB = 11.0  # If peak > this → gradient_checkpointing required

# ── Prepare model + dummy batch ───────────────────────────────────────────────
device = torch.device('cuda')

model = torchvision.models.efficientnet_b4(weights=None)
model = model.to(device, dtype=torch.float16)  # mixed precision (fp16) per config
model.train()  # training mode to include gradient computation

dummy_input = torch.randn(
    BATCH_SIZE, INPUT_C, INPUT_H, INPUT_W,
    dtype=torch.float16,
    device=device,
)
dummy_labels = torch.randint(0, 2, (BATCH_SIZE,), device=device)

# ── Peak VRAM measurement ──────────────────────────────────────────────────────
torch.cuda.reset_peak_memory_stats(device)
torch.cuda.synchronize()

# Forward pass
with torch.amp.autocast('cuda'):  # mixed precision autocast
    logits = model(dummy_input)
    loss = torch.nn.functional.cross_entropy(
        logits, dummy_labels
    )

# Backward pass
loss.backward()

torch.cuda.synchronize()
peak_bytes = torch.cuda.max_memory_allocated(device)
peak_gb    = peak_bytes / (1024 ** 3)

# ── Q3 Decision ───────────────────────────────────────────────────────────────
print('─' * 60)
print(f'Q3 VRAM Test Results')
print(f'  Batch size          : {BATCH_SIZE}')
print(f'  Input shape         : [{BATCH_SIZE}, {INPUT_C}, {INPUT_H}, {INPUT_W}]')
print(f'  Mixed precision     : fp16 (torch.cuda.amp.autocast)')
print(f'  Peak VRAM allocated : {peak_gb:.2f} GB')
print(f'  T4 Total VRAM       : {total_vram_gb:.2f} GB')
print(f'  VRAM utilisation    : {peak_gb / total_vram_gb * 100:.1f}%')
print('─' * 60)

if peak_gb > VRAM_THRESHOLD_GB:
    print(f'⚠️  Peak VRAM {peak_gb:.2f} GB > {VRAM_THRESHOLD_GB} GB threshold.')
    print('   → gradient_checkpointing=true REQUIRED (already set in config.yaml).')
    gradient_checkpointing_required = True
else:
    print(f'✅ Peak VRAM {peak_gb:.2f} GB ≤ {VRAM_THRESHOLD_GB} GB threshold.')
    print('   → gradient_checkpointing=true retained as safety precaution.')
    gradient_checkpointing_required = False

print(f'\n📋 Record in docs/adr/phase3-colab-vram.md:')
print(f'   Actual peak VRAM = {peak_gb:.2f} GB')
print(f'   gradient_checkpointing = true  (confirmed — see config.yaml)')
print(f'\n✅ Q3 RESOLVED — batch_size=16 feasible on T4 (VRAM: {peak_gb:.2f} / {total_vram_gb:.2f} GB used)')

# ── Cleanup ───────────────────────────────────────────────────────────────────
del model, dummy_input, dummy_labels, logits, loss
torch.cuda.empty_cache()
print('GPU memory released.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 4 — HuggingFace Login
# SRS: FR-CV-007 (HF Model Hub for checkpoint upload)
# Security: tokens from Colab Secrets — never hardcoded [FR-NLP-005]
# Dataset source: Google Drive (Cell 5) — Kaggle credentials no longer needed
# ─────────────────────────────────────────────────────────────────────────────

import os
from huggingface_hub import login

# ── HF token (needed for model upload in Cells 7–8) ──────────────────────────
try:
    from google.colab import userdata  # type: ignore[import]
    hf_token = userdata.get('HF_TOKEN')
except ImportError:
    hf_token = os.environ.get('HF_TOKEN')

assert hf_token, (
    'HF_TOKEN not found. '
    'Add it in Colab Secrets (Runtime → Manage Secrets) or set the HF_TOKEN env var.'
)

login(token=hf_token, add_to_git_credential=False)
print('✅ HuggingFace login successful.')
print('ℹ️  Dataset will be loaded from Google Drive in Cell 5 — no Kaggle credentials needed.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 5 — Dataset: Mount Google Drive → locate for-2seconds folder
# SRS: FR-AUD-004 (2.0 s clips), FR-CV-003 (training split)
# Source: Google Drive — folder named "for-2seconds"
# Q2 RESOLVED: for-2sec variant confirmed (2.0 s clips, 32,000 samples)
# ─────────────────────────────────────────────────────────────────────────────

import pathlib
from google.colab import drive  # type: ignore[import]

# ── Mount Google Drive ────────────────────────────────────────────────────────
drive.mount('/content/drive', force_remount=False)
print('✅ Google Drive mounted at /content/drive')

# ── Locate the for-2seconds folder ───────────────────────────────────────────
# Expected path: My Drive / for-2seconds
DRIVE_ROOT   = pathlib.Path('/content/drive/MyDrive')
FOR_2SEC_DIR = DRIVE_ROOT / 'for-2seconds'

assert FOR_2SEC_DIR.exists(), (
    f'Folder not found: {FOR_2SEC_DIR}\n'
    'Make sure your Google Drive has a folder called exactly "for-2seconds" '
    'at the root of My Drive (not inside any sub-folder).\n'
    'Expected path: My Drive/for-2seconds/training/real/*.wav'
)
print(f'✅ Dataset folder found: {FOR_2SEC_DIR}')

# ── Verify directory structure ────────────────────────────────────────────────
print('\nDirectory structure (depth 2):')
!find "{FOR_2SEC_DIR}" -maxdepth 2 -type d 2>/dev/null | head -20

print('\nFile counts per split:')
for split in ['training', 'validation', 'testing']:
    split_dir = FOR_2SEC_DIR / split
    if split_dir.exists():
        real_count = len(list((split_dir / 'real').glob('*.wav'))) if (split_dir / 'real').exists() else 0
        fake_count = len(list((split_dir / 'fake').glob('*.wav'))) if (split_dir / 'fake').exists() else 0
        total      = real_count + fake_count
        print(f'  {split:12s}: {real_count:>5,} real + {fake_count:>5,} fake = {total:>6,} total')
    else:
        print(f'  {split:12s}: NOT FOUND')

# ── Export DATA_ROOT so Cell 6 _find_for_root() resolves correctly ────────────
DATA_ROOT = FOR_2SEC_DIR.parent  # /content/drive/MyDrive

print(f'\n✅ FoR for-2sec dataset ready — Cell 6 will search under {DATA_ROOT}')
print('   Citation: Abdel-Dayem, M. (2023). Fake-or-Real (FoR) Dataset. Kaggle.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 6 — Training Loop: Two-Phase EfficientNet-B4 Training
# SRS: FR-CV-001–008 (model, training, augmentation, checkpointing, metrics)
# Modules: src/cv/model.py, src/cv/train.py
# ─────────────────────────────────────────────────────────────────────────────

import os
import sys
import pathlib

REPO_ROOT = pathlib.Path('/content/DSDBA').resolve()

# Clone the repo so the src/ package is available on Colab
if not (REPO_ROOT / 'src' / '__init__.py').exists():
    if REPO_ROOT.exists():
        !rm -rf {REPO_ROOT}
    !git clone https://github.com/zan4yov/BSDBA.git {REPO_ROOT}

assert (REPO_ROOT / 'src' / '__init__.py').exists(), (
    f'Clone failed — src/__init__.py not found at {REPO_ROOT}/src/. '
    'Verify that https://github.com/zan4yov/BSDBA is accessible.'
)
print(f'✅ Repo available at {REPO_ROOT}')

# Colab puts cwd ('') first on sys.path — that can shadow the real package.
# Always put REPO_ROOT first and chdir there so `import src` resolves correctly.
os.chdir(REPO_ROOT)
root_s = str(REPO_ROOT)
while root_s in sys.path:
    sys.path.remove(root_s)
sys.path.insert(0, root_s)

# Drop stale `src` (e.g. failed earlier import or a shadowing name in site-packages)
for _key in list(sys.modules):
    if _key == 'src' or _key.startswith('src.'):
        del sys.modules[_key]

import math
import yaml
import torch
from torch import nn
from torch.utils.data import DataLoader

from src.cv.model import DSDBAModel
from src.cv.train import (
    AudioClassificationDataset,
    build_augmentations,
    get_class_weights,
    train_epoch,
    validate_epoch,
)

# Load config as a plain dict — train.py / model.py / infer.py all expect dict,
# not the Pydantic DSDBAConfig object returned by load_config().
cfg = yaml.safe_load((REPO_ROOT / 'config.yaml').read_text())
# Override num_workers to 0 — prevents subprocess OOM kills on Colab Free Tier.
cfg['training']['num_workers'] = 0

# ── Locate FoR dataset root (auto-detect) ─────────────────────────────────────
# Cell 5 (Drive) defines DATA_ROOT = /content/drive/MyDrive.
# Keep a fallback to /content/data for older Kaggle-based flows.
if 'DATA_ROOT' in globals():
    DATA_ROOT = pathlib.Path(DATA_ROOT)  # may be a str or Path
else:
    DATA_ROOT = pathlib.Path('/content/data')

print(f'Looking for dataset under: {DATA_ROOT}')

def _find_for_root(base: pathlib.Path, _depth: int = 0) -> pathlib.Path | None:
    """Recursively find the folder that directly contains training/real/ + training/fake/."""
    if _depth > 6:
        return None
    # A valid root has training/ with at least one of real/ or fake/ inside it
    training = base / 'training'
    if training.exists() and ((training / 'real').exists() or (training / 'fake').exists()):
        return base
    # Recurse into subdirectories
    try:
        for child in sorted(base.iterdir()):
            if child.is_dir():
                result = _find_for_root(child, _depth + 1)
                if result is not None:
                    return result
    except (PermissionError, FileNotFoundError):
        pass
    return None

FOR_ROOT = _find_for_root(DATA_ROOT)

if FOR_ROOT is None:
    print('Dataset not found under /content/data.')
    print('Current contents of /content/data:')
    !find /content/data -maxdepth 3 -type d 2>/dev/null | head -30 || echo '  (empty)'
    raise FileNotFoundError(
        'FoR dataset missing. Run Cell 5 first to download it from Kaggle.\n'
        'Make sure KAGGLE_USERNAME and KAGGLE_KEY are set in Colab Secrets.'
    )

TRAIN_DIR = FOR_ROOT / 'training'
VAL_DIR   = FOR_ROOT / 'validation'
TEST_DIR  = FOR_ROOT / 'testing'

print(f'Dataset root : {FOR_ROOT}')
print('Contents:')
!find {FOR_ROOT} -maxdepth 2 -type d | head -20

def collect_files(split_dir: pathlib.Path) -> tuple[list[pathlib.Path], list[int]]:
    """Collect audio paths + labels from real/ and fake/ subdirectories."""
    paths, labels = [], []
    for label_int, label_name in [(0, 'real'), (1, 'fake')]:
        label_dir = split_dir / label_name
        if label_dir.exists():
            for f in sorted(label_dir.glob('*.wav')):
                paths.append(f)
                labels.append(label_int)
    return paths, labels

train_paths, train_labels = collect_files(TRAIN_DIR)
val_paths, val_labels     = collect_files(VAL_DIR)
test_paths, test_labels   = collect_files(TEST_DIR) if TEST_DIR.exists() else ([], [])

print(f'\nTraining samples  : {len(train_paths):,} (bonafide={train_labels.count(0):,}, spoof={train_labels.count(1):,})')
print(f'Validation samples: {len(val_paths):,} (bonafide={val_labels.count(0):,}, spoof={val_labels.count(1):,})')
print(f'Test samples      : {len(test_paths):,} (bonafide={test_labels.count(0):,}, spoof={test_labels.count(1):,})')

assert len(train_paths) > 0, (
    f'No training WAV files found under {TRAIN_DIR}.\n'
    f'Expected subfolders: {TRAIN_DIR}/real/ and {TRAIN_DIR}/fake/\n'
    'Re-run Cell 5 to re-download and re-extract the dataset.'
)
assert len(val_paths) > 0, (
    f'No validation WAV files found under {VAL_DIR}.\n'
    'Re-run Cell 5 to re-download and re-extract the dataset.'
)

# ── Build datasets — AudioClassificationDataset runs DSP on-the-fly (no cache).
# num_workers=0: single-process loading prevents subprocess OOM on Colab Free Tier.
augment   = build_augmentations(cfg)
train_ds  = AudioClassificationDataset(train_paths, train_labels, cfg, transform=augment)
val_ds    = AudioClassificationDataset(val_paths,   val_labels,   cfg, transform=None)
test_ds   = AudioClassificationDataset(test_paths,  test_labels,  cfg, transform=None) if test_paths else None

batch_size   = cfg['training']['batch_size']
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True) if test_ds else None

# ── Model setup ───────────────────────────────────────────────────────────────
OUTPUT_DIR = pathlib.Path('/content/checkpoints')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = DSDBAModel(cfg=cfg, pretrained=True).to(device)
model.freeze_backbone()

class_weights = get_class_weights(train_ds).to(device)
criterion     = nn.CrossEntropyLoss(weight=class_weights)

# ── Mixed precision scaler (config.yaml: training.mixed_precision) ────────────
use_amp = bool(cfg['training'].get('mixed_precision', False)) and device.type == 'cuda'
scaler  = torch.amp.GradScaler('cuda') if use_amp else None
print(f'Mixed precision (AMP): {"enabled" if use_amp else "disabled"}')

# ── Training hyperparameters ──────────────────────────────────────────────────
frozen_epochs = int(cfg['model'].get('frozen_epochs', 5))
max_epochs    = int(cfg['training'].get('max_epochs', 30))
finetune_lr   = float(cfg['model'].get('finetune_lr', 1e-4))
patience      = int(cfg['training'].get('early_stopping_patience', 5))

best_auc  = -math.inf
no_improve = 0
best_ckpt  = OUTPUT_DIR / 'best_model.pth'

def _save_ckpt(path, epoch, metrics):
    torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(), 'metrics': metrics}, path)

# ── Phase 1: head-only training ───────────────────────────────────────────────
print('\n── Phase 1: head-only (frozen backbone) ──')
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=min(1e-3, finetune_lr),
)
for epoch in range(1, min(frozen_epochs, max_epochs) + 1):
    tr = train_epoch(model, train_loader, optimizer, criterion, cfg, scaler=scaler)
    vl = validate_epoch(model, val_loader, cfg)
    _save_ckpt(OUTPUT_DIR / f'epoch_{epoch:02d}.pth', epoch, {**tr, **vl})
    if vl['auc_roc'] > best_auc:
        best_auc = vl['auc_roc']
        _save_ckpt(best_ckpt, epoch, {**tr, **vl})
        no_improve = 0
    else:
        no_improve += 1
    print(f'  Epoch {epoch:02d} | loss={tr["train_loss"]:.4f} acc={tr["train_acc"]:.3f} | EER={vl["eer"]:.4f} AUC={vl["auc_roc"]:.4f}')
    if no_improve >= patience:
        print(f'  Early stopping at epoch {epoch}')
        break

# ── Phase 2: fine-tune top-2 blocks ──────────────────────────────────────────
print('\n── Phase 2: fine-tuning top-2 blocks ──')
model.unfreeze_top_n(n=2)
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=min(finetune_lr, 1e-4),
)
no_improve = 0
for epoch in range(frozen_epochs + 1, max_epochs + 1):
    tr = train_epoch(model, train_loader, optimizer, criterion, cfg, scaler=scaler)
    vl = validate_epoch(model, val_loader, cfg)
    _save_ckpt(OUTPUT_DIR / f'epoch_{epoch:02d}.pth', epoch, {**tr, **vl})
    if vl['auc_roc'] > best_auc:
        best_auc = vl['auc_roc']
        _save_ckpt(best_ckpt, epoch, {**tr, **vl})
        no_improve = 0
    else:
        no_improve += 1
    print(f'  Epoch {epoch:02d} | loss={tr["train_loss"]:.4f} acc={tr["train_acc"]:.3f} | EER={vl["eer"]:.4f} AUC={vl["auc_roc"]:.4f}')
    if no_improve >= patience:
        print(f'  Early stopping at epoch {epoch}')
        break

trained_model = model
print(f'\n✅ Training complete. Best AUC-ROC={best_auc:.4f}')
print(f'Best checkpoint: {best_ckpt}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 7 — ONNX Export & Equivalence Verification
# SRS: FR-DEP-010 — |ONNX − PyTorch| < 1e-5
# Module: src/cv/infer.py
# ─────────────────────────────────────────────────────────────────────────────

from src.cv.infer import export_to_onnx, verify_onnx_equivalence

# export_to_onnx(model, cfg) -> Path  —  saves to models/checkpoints/ internally.
ONNX_PATH = export_to_onnx(trained_model, cfg)
print(f'ONNX model exported: {ONNX_PATH}')
print(f'ONNX file size: {ONNX_PATH.stat().st_size / (1024**2):.1f} MB')

# Verify numerical equivalence
equiv = verify_onnx_equivalence(trained_model, ONNX_PATH, cfg)
assert equiv, 'ONNX equivalence test FAILED — check export.'
tol = cfg['deployment']['onnx_equivalence_tolerance']
print(f'\n✅ ONNX equivalence verified: |diff| < {tol}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 8 — Upload Best Checkpoint + ONNX to HF Hub
# SRS: FR-CV-007 — model checkpoint on HuggingFace Model Hub
# MCP: huggingface-mcp (huggingface-skills)
# Security: uses hf_token from Cell 4 — never hardcoded [FR-NLP-005]
# ─────────────────────────────────────────────────────────────────────────────

from huggingface_hub import HfApi

HF_MODEL_REPO = cfg['training'].get('hf_model_repo', '')

if not HF_MODEL_REPO:
    print('⚠️  config.yaml:training.hf_model_repo is empty. Skipping upload.')
else:
    api = HfApi(token=hf_token)
    api.create_repo(repo_id=HF_MODEL_REPO, repo_type='model', exist_ok=True)

    api.upload_file(
        path_or_fileobj=str(best_ckpt),
        path_in_repo='best_model.pth',
        repo_id=HF_MODEL_REPO,
        repo_type='model',
    )
    print(f'✅ Best checkpoint uploaded to {HF_MODEL_REPO}/best_model.pth')

    api.upload_file(
        path_or_fileobj=str(ONNX_PATH),
        path_in_repo='dsdba_efficientnet_b4.onnx',
        repo_id=HF_MODEL_REPO,
        repo_type='model',
    )
    print(f'✅ ONNX model uploaded to {HF_MODEL_REPO}/dsdba_efficientnet_b4.onnx')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 9 — Final Evaluation: EER & AUC-ROC Verification
# SRS: FR-CV-008 — EER ≤ 10%, AUC-ROC ≥ 0.90 on FoR TEST set
# Uses test_loader (testing/ split) for the binding acceptance gate.
# Falls back to val_loader if testing/ split was not present in the dataset.
# ─────────────────────────────────────────────────────────────────────────────

import torch
from src.cv.train import validate_epoch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
trained_model.to(device)

eval_loader = test_loader if test_loader is not None else val_loader
eval_split  = 'test' if test_loader is not None else 'validation (test split not found)'
print(f'Evaluating on: {eval_split}')

# validate_epoch(model, loader, cfg: dict) -> {"eer": float, "auc_roc": float}
eval_results = validate_epoch(trained_model, eval_loader, cfg)

target_auc = cfg['acceptance_criteria']['target_auc_roc']
target_eer = cfg['acceptance_criteria']['target_eer']

print('=' * 60)
print(f'  SPRINT B — FINAL EVALUATION RESULTS  [{eval_split.upper()}]')
print('=' * 60)
print(f'  AUC-ROC : {eval_results["auc_roc"]:.4f}  (target >= {target_auc})')
print(f'  EER     : {eval_results["eer"]:.4f}  (target <= {target_eer})')
print('=' * 60)

auc_ok = eval_results['auc_roc'] >= target_auc
eer_ok = eval_results['eer']     <= target_eer

if auc_ok:
    print(f'  PASS  AUC-ROC >= {target_auc}')
else:
    print(f'  FAIL  AUC-ROC < {target_auc} (continue tuning)')

if eer_ok:
    print(f'  PASS  EER <= {target_eer}')
else:
    print(f'  FAIL  EER > {target_eer} (continue tuning)')

print()
if auc_ok and eer_ok:
    print('Sprint B acceptance criteria MET. Proceed to Sprint C (Grad-CAM).')
else:
    print('Acceptance criteria not yet met. Consider:')
    print('  1. Increase max_epochs in config.yaml')
    print('  2. Adjust finetune_lr')
    print('  3. Verify dataset quality and split balance')